# Fine-tuning LLM w Google Colab: LoRA / PEFT na modelu Qwen 0.5B

Ten notebook pokazuje praktyczny, dydaktyczny przykład **supervised fine-tuning** małego modelu instrukcyjnego.

Używamy:

- `Qwen/Qwen2.5-0.5B-Instruct` jako modelu bazowego,
- `PEFT / LoRA`, aby nie trenować wszystkich wag modelu,
- mini-zbioru danych uczącego model odpowiadania w określonym formacie,
- standardowego `Trainer` z Hugging Face.

## Cel zajęć

Student ma zrozumieć różnicę między:

1. zwykłym promptowaniem,
2. RAG,
3. fine-tuningiem,
4. treningiem modelu od zera.

Ten notebook **nie trenuje modelu od zera**. Robi lekkie dostrojenie przez LoRA, czyli modyfikuje niewielką liczbę dodatkowych parametrów.

## Zalecane ustawienie Colab

`Runtime → Change runtime type → T4 GPU`

Na CPU notebook może działać bardzo wolno.


# 1. Instalacja bibliotek

Instalujemy podstawowy zestaw:

- `transformers` — ładowanie modelu i tokenizera,
- `datasets` — budowa zbioru treningowego,
- `accelerate` — obsługa treningu na GPU,
- `peft` — LoRA i inne metody parameter-efficient fine-tuning.


In [ ]:
!pip -q install -U transformers datasets accelerate peft

# 2. Importy i sprawdzenie środowiska

In [ ]:
import os
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("GPU dostępne:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# 3. Model bazowy

Używamy małego modelu instrukcyjnego:

`Qwen/Qwen2.5-0.5B-Instruct`

To model wystarczająco mały, żeby pokazać fine-tuning w Colabie, ale nadal będący prawdziwym modelem językowym typu causal language model.

Ważne:

- nie trenujemy wszystkich wag,
- dokładamy adapter LoRA,
- uczymy tylko niewielką liczbę parametrów.


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Modele przyczynowe często nie mają osobnego tokena PAD.
# Do treningu ustawiamy PAD na EOS.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

print("Model załadowany:", BASE_MODEL)
print("Urządzenie:", device)


# 4. Funkcja generująca odpowiedź przed fine-tuningiem

Najpierw sprawdzimy, jak zachowuje się model bazowy.

Potem wykonamy fine-tuning i zadamy podobne pytania jeszcze raz.

Dzięki temu studenci zobaczą różnicę:

```text
model bazowy → model po dostrojeniu
```


In [ ]:
def generate_answer(prompt, model, tokenizer, max_new_tokens=220):
    model.eval()

    messages = [
        {
            "role": "system",
            "content": "Jesteś asystentem AI. Odpowiadasz po polsku."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## Test przed fine-tuningiem

Chcemy nauczyć model odpowiadania w bardzo konkretnym formacie:

```text
DECYZJA:
UZASADNIENIE:
NASTĘPNY KROK:
RYZYKO:
```

Przed dostrojeniem model może nie trzymać tego formatu konsekwentnie.


In [ ]:
test_prompts = [
    "Mam dokument PDF z regulaminem studiów i chcę zadawać pytania o jego treść. Jaką ścieżkę AI wybrać?",
    "Chcę, żeby model zawsze pisał w stylu formalnej opinii prawnej. Czy potrzebuję RAG czy fine-tuningu?",
    "Mam plik CSV ze sprzedażą i chcę policzyć średnią oraz zrobić wykres. Jaką ścieżkę AI wybrać?"
]

for p in test_prompts:
    print("=" * 100)
    print("PYTANIE:", p)
    print("\nODPOWIEDŹ MODELU BAZOWEGO:")
    print(generate_answer(p, model, tokenizer))


# 5. Mini-zbiór danych do fine-tuningu

Budujemy mały zbiór instrukcji.

Każdy przykład ma:

- `instruction` — pytanie użytkownika,
- `answer` — oczekiwaną odpowiedź w narzuconym formacie.

To jest miniaturowy przykład **supervised fine-tuning**.

W prawdziwym projekcie danych powinno być znacznie więcej: setki, tysiące albo dziesiątki tysięcy przykładów. Tutaj robimy wersję dydaktyczną.


In [ ]:
training_examples = [
    {'instruction': 'Mam dokument PDF z regulaminem studiów i chcę zadawać pytania o jego treść. Jaką ścieżkę AI wybrać?', 'answer': 'DECYZJA: RAG\nUZASADNIENIE: Pytanie wymaga pracy na konkretnym dokumencie, więc model powinien otrzymać fragmenty regulaminu jako kontekst.\nNASTĘPNY KROK: Podziel dokument na fragmenty, utwórz embeddingi i podłącz bazę wektorową.\nRYZYKO: Sam LLM bez dokumentów może halucynować.'},
    {'instruction': 'Chcę zapytać model, czym jest transformer. Jaką ścieżkę AI wybrać?', 'answer': 'DECYZJA: PROMPT\nUZASADNIENIE: To pytanie dotyczy wiedzy ogólnej, więc zwykły prompt powinien wystarczyć.\nNASTĘPNY KROK: Przygotuj jasne pytanie i określ poziom odbiorcy.\nRYZYKO: Odpowiedź może być zbyt ogólna, jeśli prompt nie określi kontekstu.'},
    {'instruction': 'Mam plik CSV ze sprzedażą i chcę policzyć średnią oraz zrobić wykres. Jaką ścieżkę AI wybrać?', 'answer': 'DECYZJA: TOOL\nUZASADNIENIE: Zadanie wymaga obliczeń i operacji na danych, więc potrzebne jest narzędzie, np. Python.\nNASTĘPNY KROK: Wczytaj CSV w pandas, wykonaj obliczenia i wygeneruj wykres.\nRYZYKO: LLM bez wykonania kodu może podać błędne liczby.'},
    {'instruction': 'Chcę, żeby model zawsze pisał w stylu formalnej opinii prawnej. Czy potrzebuję RAG czy fine-tuningu?', 'answer': 'DECYZJA: FINE_TUNING\nUZASADNIENIE: Chodzi o utrwalenie stylu i formatu odpowiedzi, a nie tylko o dostęp do dokumentów.\nNASTĘPNY KROK: Najpierw przetestuj system prompt i przykłady, a dopiero potem rozważ fine-tuning.\nRYZYKO: Fine-tuning na zbyt małej liczbie przykładów może pogorszyć model.'},
    {'instruction': 'Mam bazę procedur HR i chcę, żeby pracownicy zadawali pytania o urlopy. Co wybrać?', 'answer': 'DECYZJA: RAG\nUZASADNIENIE: Odpowiedzi zależą od konkretnych procedur, które mogą się zmieniać.\nNASTĘPNY KROK: Zbuduj bazę dokumentów, embeddingi i mechanizm cytowania źródeł.\nRYZYKO: Bez aktualnych dokumentów model może podać nieaktualne zasady.'},
    {'instruction': 'Chcę nauczyć model rozpoznawania kategorii wiadomości klientów. Co wybrać?', 'answer': 'DECYZJA: FINE_TUNING\nUZASADNIENIE: To zadanie klasyfikacyjne, w którym model może uczyć się na oznaczonych przykładach.\nNASTĘPNY KROK: Przygotuj zbiór wiadomości z poprawnymi etykietami.\nRYZYKO: Dane treningowe muszą być reprezentatywne, inaczej model będzie źle generalizował.'},
    {'instruction': 'Student pyta, kiedy trzeba oddać projekt według sylabusa. Co wybrać?', 'answer': 'DECYZJA: RAG\nUZASADNIENIE: Termin oddania projektu pochodzi z dokumentu, więc model musi odwołać się do sylabusa.\nNASTĘPNY KROK: Dodaj sylabus do bazy wiedzy i wymuś odpowiedzi na podstawie kontekstu.\nRYZYKO: Bez dokumentu model może zmyślić termin.'},
    {'instruction': 'Użytkownik pyta, czym różni się RAG od fine-tuningu. Co wybrać?', 'answer': 'DECYZJA: PROMPT\nUZASADNIENIE: To pytanie koncepcyjne, które nie wymaga dokumentów ani obliczeń.\nNASTĘPNY KROK: Poproś model o porównanie w tabeli lub przykładach.\nRYZYKO: Odpowiedź może uprościć różnice, jeśli prompt będzie zbyt ogólny.'},
    {'instruction': 'Mam tabelę ocen studentów i chcę znaleźć osoby zagrożone brakiem zaliczenia. Co wybrać?', 'answer': 'DECYZJA: TOOL\nUZASADNIENIE: To analiza danych, która wymaga obliczeń i filtrowania.\nNASTĘPNY KROK: Użyj pandas do obliczenia średnich, progów i warunków zaliczenia.\nRYZYKO: Model językowy bez narzędzia może pomylić obliczenia.'},
    {'instruction': 'Chcę, aby asystent firmy odpowiadał zawsze według schematu: decyzja, uzasadnienie, ryzyko. Co wybrać?', 'answer': 'DECYZJA: FINE_TUNING\nUZASADNIENIE: Chodzi o stabilny format odpowiedzi, który można utrwalić na przykładach.\nNASTĘPNY KROK: Przygotuj kilkaset przykładów odpowiedzi w docelowym schemacie.\nRYZYKO: Najpierw warto sprawdzić, czy nie wystarczy dobry system prompt.'},
    {'instruction': 'Chcę zadać pytanie o aktualne saldo klienta w systemie CRM. Co wybrać?', 'answer': 'DECYZJA: TOOL\nUZASADNIENIE: Aktualne saldo musi pochodzić z systemu lub API, nie z pamięci modelu.\nNASTĘPNY KROK: Podłącz bezpieczne API CRM albo bazę danych.\nRYZYKO: Model bez dostępu do systemu poda fikcyjną odpowiedź.'},
    {'instruction': 'Mam instrukcję BHP i chcę pytać, jakie środki ochrony są wymagane. Co wybrać?', 'answer': 'DECYZJA: RAG\nUZASADNIENIE: Odpowiedź musi wynikać z konkretnej instrukcji BHP.\nNASTĘPNY KROK: Załaduj dokument, wykonaj chunking i dodaj cytowanie fragmentów.\nRYZYKO: Halucynacje w obszarze BHP mogą być niebezpieczne.'},
    {'instruction': 'Chcę, aby model pisał krótkie odpowiedzi w tonie eksperta technicznego. Co wybrać?', 'answer': 'DECYZJA: PROMPT\nUZASADNIENIE: Styl można najpierw kontrolować instrukcją systemową i przykładami.\nNASTĘPNY KROK: Zdefiniuj ton, długość i strukturę odpowiedzi w system prompt.\nRYZYKO: Fine-tuning byłby przedwczesny, jeśli prosty prompt działa dobrze.'},
    {'instruction': 'Chcę, żeby model rozpoznawał intencję użytkownika: RAG, TOOL, PROMPT albo FINE_TUNING. Co wybrać?', 'answer': 'DECYZJA: FINE_TUNING\nUZASADNIENIE: To powtarzalne zadanie klasyfikacyjne, które można nauczyć na oznaczonych przykładach.\nNASTĘPNY KROK: Przygotuj zróżnicowany zbiór promptów z poprawnymi etykietami.\nRYZYKO: Zbyt mały zbiór danych da niestabilny klasyfikator.'},
    {'instruction': 'Czy do odpowiedzi na pytanie o regulamin firmy wystarczy prompt?', 'answer': 'DECYZJA: RAG\nUZASADNIENIE: Regulamin firmy jest konkretnym źródłem wiedzy, więc odpowiedź powinna opierać się na dokumentach.\nNASTĘPNY KROK: Podłącz aktualny regulamin jako źródło w systemie RAG.\nRYZYKO: Prompt bez dokumentu może dać poprawnie brzmiącą, ale fałszywą odpowiedź.'},
    {'instruction': 'Czy do streszczenia ogólnego artykułu o AI potrzebuję fine-tuningu?', 'answer': 'DECYZJA: PROMPT\nUZASADNIENIE: Jednorazowe streszczenie zwykle nie wymaga dostrajania modelu.\nNASTĘPNY KROK: Wklej tekst i określ długość oraz styl streszczenia.\nRYZYKO: Jeśli tekst jest długi, trzeba kontrolować kontekst albo zastosować chunking.'},
    {'instruction': 'Mam faktury w PDF i chcę wyciągnąć kwoty netto, VAT i brutto. Co wybrać?', 'answer': 'DECYZJA: TOOL\nUZASADNIENIE: Potrzebne jest przetwarzanie dokumentów i ekstrakcja danych, najlepiej przez OCR/parser oraz Python.\nNASTĘPNY KROK: Wydobądź tekst/tabele, a potem użyj walidacji obliczeń.\nRYZYKO: Sam LLM może pomylić kwoty albo formaty liczb.'},
    {'instruction': 'Chcę odpowiadać klientom zgodnie z najnowszą ofertą produktową. Co wybrać?', 'answer': 'DECYZJA: RAG\nUZASADNIENIE: Oferta produktowa jest zmienna, więc powinna być pobierana z aktualnej bazy wiedzy.\nNASTĘPNY KROK: Podłącz katalog produktów, cenniki i FAQ do RAG.\nRYZYKO: Fine-tuning utrwaliłby wiedzę, która szybko się starzeje.'},
    {'instruction': 'Chcę stworzyć model, który stale odpowiada głosem konkretnej marki. Co wybrać?', 'answer': 'DECYZJA: FINE_TUNING\nUZASADNIENIE: Stały głos marki i powtarzalny styl mogą być utrwalane przez dostrajanie na przykładach.\nNASTĘPNY KROK: Zbierz dobre i złe przykłady komunikacji marki.\nRYZYKO: Bez jasnych standardów stylu fine-tuning nauczy chaosu.'},
    {'instruction': 'Użytkownik pyta, jaka jest różnica między embeddingiem a tokenem. Co wybrać?', 'answer': 'DECYZJA: PROMPT\nUZASADNIENIE: To pytanie edukacyjne i nie wymaga bazy dokumentów.\nNASTĘPNY KROK: Poproś model o definicje i prosty przykład.\nRYZYKO: Odpowiedź może być zbyt techniczna bez określenia poziomu odbiorcy.'},
]

dataset = Dataset.from_list(training_examples)
dataset


In [ ]:
dataset[0]

# 6. Formatowanie danych do stylu rozmowy

Modele instrukcyjne uczymy zwykle na formacie rozmowy:

```text
system → user → assistant
```

Zmienimy każdy przykład na tekst zgodny z chat template modelu.


In [ ]:
SYSTEM_PROMPT = (
    "Jesteś routerem workflow AI. "
    "Zawsze odpowiadasz po polsku w dokładnie czterech polach: "
    "DECYZJA, UZASADNIENIE, NASTĘPNY KROK, RYZYKO."
)

def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["answer"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0]["text"])


# 7. Dodanie LoRA do modelu

LoRA pozwala dostroić model bez aktualizowania wszystkich jego wag.

Zamiast trenować pełny model, dokładamy małe macierze adaptacyjne do wybranych warstw.

W tym przykładzie używamy typowych modułów projekcyjnych modelu transformerowego:

- `q_proj`,
- `k_proj`,
- `v_proj`,
- `o_proj`,
- `gate_proj`,
- `up_proj`,
- `down_proj`.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# 8. Tokenizacja danych treningowych

Zmieniamy tekst na tokeny.

Dla prostoty uczymy model przewidywania kolejnych tokenów całego przykładu rozmowy.

W bardziej zaawansowanej wersji można maskować część `user` i liczyć stratę tylko na odpowiedzi `assistant`.


In [ ]:
MAX_LENGTH = 512

def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    remove_columns=formatted_dataset.column_names
)

print(tokenized_dataset[0].keys())
print("Liczba przykładów:", len(tokenized_dataset))


# 9. Trening LoRA

To jest właściwy fine-tuning.

Parametry są celowo małe, żeby notebook działał szybko na Colabie.

Jeżeli chcesz mocniejszy efekt, możesz zwiększyć:

- `max_steps`,
- liczbę przykładów,
- `r` w LoRA,
- jakość danych.


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./qwen_lora_router",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=60,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()


# 10. Test po fine-tuningu

Teraz sprawdzamy, czy model lepiej trzyma narzucony format odpowiedzi.


In [ ]:
after_prompts = [
    "Mam dokument PDF z regulaminem studiów i chcę zadawać pytania o jego treść. Jaką ścieżkę AI wybrać?",
    "Chcę, żeby model zawsze pisał w stylu formalnej opinii prawnej. Czy potrzebuję RAG czy fine-tuningu?",
    "Mam plik CSV ze sprzedażą i chcę policzyć średnią oraz zrobić wykres. Jaką ścieżkę AI wybrać?",
    "Chcę zadać pytanie o aktualną cenę produktu w sklepie internetowym. Co wybrać?",
    "Chcę wyjaśnić studentom, czym jest embedding. Co wybrać?"
]

for p in after_prompts:
    print("=" * 100)
    print("PYTANIE:", p)
    print("\nODPOWIEDŹ PO FINE-TUNINGU:")
    print(generate_answer(p, model, tokenizer))


# 11. Zapis adaptera LoRA

Nie zapisujemy pełnego modelu. Zapisujemy tylko adapter LoRA.

To jest jedna z głównych zalet PEFT:

- mały rozmiar,
- łatwe przenoszenie,
- można podłączyć adapter do modelu bazowego.


In [ ]:
ADAPTER_DIR = "./qwen_lora_router_adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter zapisany w:", ADAPTER_DIR)
print("Pliki:", os.listdir(ADAPTER_DIR))


# 12. Spakowanie adaptera do ZIP

Ten plik można pobrać z Colaba i później wczytać razem z modelem bazowym.


In [ ]:
import shutil

zip_path = shutil.make_archive("qwen_lora_router_adapter", "zip", ADAPTER_DIR)
print("Utworzono:", zip_path)


# 13. Jak wczytać adapter później

Poniższa komórka pokazuje schemat ponownego użycia adaptera.

Możesz ją uruchomić po zapisaniu adaptera albo w osobnym notebooku, jeżeli masz folder adaptera.


In [ ]:
# Opcjonalne: ponowne wczytanie modelu bazowego i adaptera LoRA

# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL,
#     torch_dtype=dtype,
#     trust_remote_code=True,
# ).to(device)

# loaded_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(device)
# loaded_model.eval()

# print(generate_answer(
#     "Mam regulamin firmy i chcę odpowiadać na pytania pracowników. Co wybrać?",
#     loaded_model,
#     tokenizer
# ))


# 14. Co student powinien zrozumieć

## RAG

RAG jest dobry, gdy model musi odpowiadać na podstawie dokumentów, regulaminów, procedur, cenników, FAQ lub wiedzy, która się zmienia.

## Fine-tuning

Fine-tuning jest dobry, gdy chcemy utrwalić:

- styl,
- format,
- klasyfikację,
- sposób reagowania,
- specyficzny typ odpowiedzi.

## Tool use

Narzędzia są potrzebne, gdy trzeba wykonać:

- obliczenia,
- analizę danych,
- zapytanie SQL,
- użycie API,
- generowanie wykresów.

## Prompt

Prompt wystarcza przy wielu zadaniach ogólnych i edukacyjnych.

Największy błąd początkujących:

> Trenować model wtedy, gdy wystarczyłby RAG, prompt albo Python.


# 15. Zadania dla studentów

## Zadanie 1

Dodaj 10 nowych przykładów treningowych dla kategorii `RAG`.

## Zadanie 2

Dodaj 10 przykładów dla kategorii `TOOL`.

## Zadanie 3

Zmień format odpowiedzi na:

```text
KATEGORIA:
DLACZEGO:
IMPLEMENTACJA:
KONTROLA JAKOŚCI:
```

i wykonaj ponowny fine-tuning.

## Zadanie 4

Porównaj odpowiedzi modelu:

1. przed fine-tuningiem,
2. po 20 krokach,
3. po 60 krokach,
4. po 150 krokach.

## Zadanie 5

Wyjaśnij, dlaczego ten notebook nie jest treningiem modelu od zera.


# 16. Puenta prowadzącego

Fine-tuning nie służy do wszystkiego.

Jeżeli wiedza często się zmienia, lepszy jest RAG.

Jeżeli trzeba liczyć, lepszy jest Python.

Jeżeli chodzi o jednorazowe wyjaśnienie, wystarczy prompt.

Fine-tuning ma sens wtedy, gdy chcemy nauczyć model **powtarzalnego zachowania**: formatu, stylu, klasyfikacji lub sposobu odpowiedzi.

```text
Prompt daje instrukcję.
RAG daje pamięć.
Tool daje działanie.
Fine-tuning daje nawyk.
```
